# Users DB → partitioned search index (cloud-only, no localhost)

Builds the searchable parts from `users_data.zip` and uploads each part to its
own account. See `PARTS_README.md` for the full design.

1. Run the setup cell.
2. Fill in the **config** cell (source link, number of parts, one upload command per account).
3. Run the build cell.
4. Put the resulting base URLs into `parts-edge.js` `DEFAULT_PART_BASES` and deploy `edgeone-parts-deploy.zip`.

In [ ]:
!apt-get -qq install -y gh >/dev/null 2>&1
!rm -rf users-shards 2>/dev/null; git clone -q https://github.com/namanyash2009-crypto/users-shards.git
%cd users-shards
print('ready')

In [ ]:
import os, json

# --- paste a FRESH direct link to users_data.zip (must support HTTP Range) ---
SOURCE = 'https://flowvideoplayer.com/api/v2/download/direct?url=https%3A%2F%2F1024terabox.com%2Fs%2F1ahJz-qdH7h_9One0lXxDoA&idx=0'

PARTS = 10            # = number of accounts you will use
EST_TOTAL = 1710000000

# One upload command per part (index 0 -> part-000).
# GitHub example (create the releases first):
#   UPLOAD_CMDS = [f'gh release upload v1 {{dir}}/* --repo ACCT{i}/REPO --clobber' for i in range(PARTS)]
# First authenticate each account token:  os.system('echo TOKEN | gh auth login --with-token')
# Internet Archive example (needs `ia` configured per account):
#   UPLOAD_CMDS = [f'ia upload users-db-part-{i:03d} {{dir}}/*' for i in range(PARTS)]
UPLOAD_CMDS = []      # leave empty to just write ./out/part-XXX locally

os.environ.update(
    SOURCE=SOURCE, PARTS=str(PARTS), EST_TOTAL=str(EST_TOTAL),
    MOB_BUCKETS='64', MAIL_BUCKETS='64', NAME_BUCKETS='64', BLOCK='2048',
    OUT='./out', GZIP_STAGE='1', DELETE_AFTER_UPLOAD='1' if UPLOAD_CMDS else '0',
)
if UPLOAD_CMDS:
    os.environ['UPLOAD_CMDS'] = json.dumps(UPLOAD_CMDS)
print('source:', SOURCE[:70], '...')
print('parts :', PARTS, '| upload cmds:', len(UPLOAD_CMDS))

In [ ]:
!python parts_builder.py

When it finishes, each `out/part-XXX` has been uploaded (and deleted locally if
`DELETE_AFTER_UPLOAD=1`). Collect the base URLs, e.g.:

* GitHub: `https://github.com/ACCT/repo/releases/download/v1`
* Internet Archive: `https://archive.org/download/ITEM`

Then set them in `parts-edge.js` `DEFAULT_PART_BASES` **in part order** and deploy.